# SaaS Revenue Intelligence Platform
## PySpark Revenue Analysis — Databricks Notebook
**Author:** Kritheshvar Vinothkumar
**Stack:** Databricks · PySpark · Snowflake GOLD · Delta Lake

In [ ]:
%pip install snowflake-connector-python pandas matplotlib seaborn --quiet

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
from datetime import datetime

SNOWFLAKE_OPTIONS = {
    'sfURL':       'tejxpcq-vw92165.snowflakecomputing.com',
    'sfUser':      dbutils.secrets.get(scope='saas-scope', key='snowflake-user'),
    'sfPassword':  dbutils.secrets.get(scope='saas-scope', key='snowflake-password'),
    'sfDatabase':  'SAAS_REVENUE_DB',
    'sfWarehouse': 'SAAS_WH',
    'sfSchema':    'GOLD',
    'sfRole':      'SYSADMIN',
}
SNOWFLAKE_SOURCE = 'net.snowflake.spark.snowflake'
print('Connected to Snowflake: tejxpcq-vw92165 / SAAS_REVENUE_DB / GOLD')

In [ ]:
# Load MRR data from Snowflake GOLD
def read_snowflake(query):
    return (spark.read.format(SNOWFLAKE_SOURCE)
            .options(**SNOWFLAKE_OPTIONS)
            .option('query', query).load())

df_mrr = read_snowflake('''
    SELECT SNAPSHOT_DATE, CUSTOMER_ID, PLAN_TYPE,
           MRR_AMOUNT, ARR_AMOUNT, IS_CHURNED,
           COHORT_MONTH, COUNTRY
    FROM SAAS_REVENUE_DB.GOLD.FACT_MRR_SNAPSHOT
    WHERE SNAPSHOT_DATE >= DATEADD('month', -12, CURRENT_DATE())
''').cache()

print(f'Loaded {df_mrr.count():,} rows from GOLD.FACT_MRR_SNAPSHOT')
df_mrr.printSchema()

In [ ]:
# MRR Trend — month by month
window_lag = Window.orderBy('MONTH')
df_monthly_mrr = (
    df_mrr.filter(F.col('IS_CHURNED') == False)
    .withColumn('MONTH', F.date_trunc('month', F.col('SNAPSHOT_DATE')))
    .groupBy('MONTH')
    .agg(
        F.sum('MRR_AMOUNT').alias('TOTAL_MRR'),
        F.countDistinct('CUSTOMER_ID').alias('ACTIVE_CUSTOMERS'),
        F.avg('MRR_AMOUNT').alias('AVG_MRR_PER_CUSTOMER'),
    )
    .withColumn('MRR_GROWTH_PCT',
        F.round((F.col('TOTAL_MRR') - F.lag('TOTAL_MRR',1).over(window_lag))
                / F.lag('TOTAL_MRR',1).over(window_lag) * 100, 2))
    .orderBy('MONTH')
)
display(df_monthly_mrr)

In [ ]:
# Plot MRR trend
pdf_mrr = df_monthly_mrr.toPandas()
pdf_mrr['MONTH'] = pd.to_datetime(pdf_mrr['MONTH'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('Monthly Recurring Revenue Trend', fontsize=14, fontweight='bold')

ax1.fill_between(pdf_mrr['MONTH'], pdf_mrr['TOTAL_MRR'], alpha=0.3, color='#2196F3')
ax1.plot(pdf_mrr['MONTH'], pdf_mrr['TOTAL_MRR'], color='#2196F3', linewidth=2, marker='o')
ax1.set_ylabel('Total MRR ($)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.grid(axis='y', alpha=0.3)

colors = ['#4CAF50' if v >= 0 else '#F44336' for v in pdf_mrr['MRR_GROWTH_PCT'].fillna(0)]
ax2.bar(pdf_mrr['MONTH'], pdf_mrr['MRR_GROWTH_PCT'].fillna(0), color=colors, width=20)
ax2.axhline(0, color='gray', linewidth=0.8)
ax2.set_ylabel('MoM Growth (%)')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ARR by Plan Type
latest_month = df_mrr.agg(F.max('SNAPSHOT_DATE')).collect()[0][0]

df_arr_by_plan = (
    df_mrr
    .filter((F.col('SNAPSHOT_DATE') == latest_month) & (F.col('IS_CHURNED') == False))
    .groupBy('PLAN_TYPE')
    .agg(
        F.sum('ARR_AMOUNT').alias('TOTAL_ARR'),
        F.countDistinct('CUSTOMER_ID').alias('CUSTOMER_COUNT'),
        F.avg('MRR_AMOUNT').alias('AVG_MRR'),
    )
    .orderBy(F.col('TOTAL_ARR').desc())
)
display(df_arr_by_plan)

In [ ]:
# Churn Rate Analysis
df_churn = (
    df_mrr
    .withColumn('MONTH', F.date_trunc('month', F.col('SNAPSHOT_DATE')))
    .groupBy('MONTH')
    .agg(
        F.countDistinct('CUSTOMER_ID').alias('TOTAL_CUSTOMERS'),
        F.countDistinct(
            F.when(F.col('IS_CHURNED') == True, F.col('CUSTOMER_ID'))
        ).alias('CHURNED_CUSTOMERS'),
        F.sum(F.when(F.col('IS_CHURNED') == True, F.col('MRR_AMOUNT')).otherwise(0)).alias('CHURNED_MRR'),
    )
    .withColumn('CHURN_RATE_PCT',
        F.round(F.col('CHURNED_CUSTOMERS') / F.col('TOTAL_CUSTOMERS') * 100, 2))
    .orderBy('MONTH')
)
display(df_churn)

In [ ]:
# LTV Estimation by Plan
avg_churn_rate = df_churn.agg(F.avg('CHURN_RATE_PCT')).collect()[0][0] / 100
print(f'Avg monthly churn rate: {avg_churn_rate*100:.2f}%')

df_ltv = (
    df_mrr
    .filter((F.col('SNAPSHOT_DATE') == latest_month) & (F.col('IS_CHURNED') == False))
    .groupBy('PLAN_TYPE')
    .agg(F.avg('MRR_AMOUNT').alias('AVG_MRR'),
         F.countDistinct('CUSTOMER_ID').alias('CUSTOMER_COUNT'))
    .withColumn('EST_LTV', F.round(F.col('AVG_MRR') / F.lit(avg_churn_rate), 2))
    .withColumn('AVG_LIFETIME_MONTHS', F.round(F.lit(1) / F.lit(avg_churn_rate), 1))
    .orderBy(F.col('EST_LTV').desc())
)
display(df_ltv)

In [ ]:
# Cohort Retention Heatmap
df_cohort = (
    df_mrr
    .withColumn('MONTH', F.date_trunc('month', F.col('SNAPSHOT_DATE')))
    .withColumn('MONTHS_SINCE_JOIN',
        F.months_between(F.col('MONTH'), F.col('COHORT_MONTH')).cast(IntegerType()))
    .groupBy('COHORT_MONTH', 'MONTHS_SINCE_JOIN')
    .agg(F.countDistinct('CUSTOMER_ID').alias('ACTIVE_CUSTOMERS'))
    .orderBy('COHORT_MONTH', 'MONTHS_SINCE_JOIN')
)

pdf_cohort = df_cohort.toPandas()
pdf_cohort['COHORT_MONTH'] = pd.to_datetime(pdf_cohort['COHORT_MONTH']).dt.strftime('%Y-%m')

pivot = pdf_cohort.pivot(index='COHORT_MONTH', columns='MONTHS_SINCE_JOIN', values='ACTIVE_CUSTOMERS')
pivot_pct = pivot.div(pivot[0], axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot_pct, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Retention %'})
ax.set_title('Customer Cohort Retention Heatmap (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Months Since First Subscription')
ax.set_ylabel('Cohort (Signup Month)')
plt.tight_layout()
plt.show()

In [ ]:
# Write all results back to Snowflake GOLD
def write_to_snowflake(df, table_name, mode='overwrite'):
    (df.write.format(SNOWFLAKE_SOURCE)
     .options(**SNOWFLAKE_OPTIONS)
     .option('dbtable', table_name)
     .mode(mode).save())
    print(f'Written to GOLD.{table_name}')

run_ts = F.lit(datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'))
write_to_snowflake(df_monthly_mrr.withColumn('COMPUTED_AT', run_ts), 'AGG_MONTHLY_MRR')
write_to_snowflake(df_arr_by_plan.withColumn('COMPUTED_AT', run_ts), 'AGG_ARR_BY_PLAN')
write_to_snowflake(df_churn.withColumn('COMPUTED_AT', run_ts),       'AGG_MONTHLY_CHURN')
write_to_snowflake(df_ltv.withColumn('COMPUTED_AT', run_ts),         'AGG_LTV_BY_PLAN')
print('All aggregations written to Snowflake GOLD!')

In [ ]:
# KPI Summary
latest_mrr  = pdf_mrr['TOTAL_MRR'].iloc[-1]
avg_churn   = pdf_churn['CHURN_RATE_PCT'].mean() if 'pdf_churn' in dir() else 0
total_custs = pdf_mrr['ACTIVE_CUSTOMERS'].iloc[-1]
mrr_growth  = pdf_mrr['MRR_GROWTH_PCT'].iloc[-1]

print('=' * 48)
print('  SaaS Revenue Intelligence — KPI Summary')
print('=' * 48)
print(f'  Current MRR       : ${latest_mrr:>12,.2f}')
print(f'  Current ARR       : ${latest_mrr*12:>12,.2f}')
print(f'  Active Customers  : {total_custs:>13,}')
print(f'  MoM MRR Growth    : {mrr_growth:>12.1f}%')
print(f'  Avg Monthly Churn : {avg_churn:>12.2f}%')
print('=' * 48)